# DEM correction + 30 m resampling of the SEAS5 forecast (Kakamega, Kilifi, Nyeri)
Takes the daily forecast GeoTIFFs (0.05°, made from the ~1° SEAS5 grid) and puts them on a **30 m grid** using your high-resolution DEM.

| Variable | What is done |
|---|---|
| **Temperature** | Resampled to 30 m, then `T + lapse × (DEM − smoothed reference DEM)` (default −6.5 °C/km) |
| **RH** | Dewpoint is recovered from T and RH (inverse Magnus), T and dewpoint are corrected separately (−6.5 and −2.0 °C/km), RH is recomputed from the two |
| **Tmax / tmin** | Resampled to 30 m, then the same lapse correction as the mean temperature (`T + lapse × (DEM − smoothed reference DEM)`). Nyeri is re-centred on the fly (`RECENTRE_COUNTIES`) |
| **Rainfall** | Bilinear resampling only (the CHIRPS wetter/drier pattern is switched off, `USE_CHIRPS_PATTERN = False`) |
| **Wind speed** | Bilinear resampling only |

**Honest limits:** only temperature and RH gain real terrain detail. Rain and wind are resampled to match the 30 m grid and carry no new information. The forecast signal itself (warm/cool, wet/dry vs normal) stays county-scale.

**Outputs:** `OUT_DIR/<group>/<var>/<stat>/<County>_<var>_<stat>_<YYYY-MM-DD>.tif`, same naming as the inputs.

**Setup:** `pip install rasterio rioxarray xarray scipy matplotlib pandas`

In [ ]:
import os, re, glob, gc
import numpy as np, pandas as pd
import rasterio, rioxarray as rxr
from rasterio.enums import Resampling
from rasterio.warp import reproject
from scipy import ndimage
import matplotlib.pyplot as plt
# ---------------- your settings ----------------
COUNTIES   = [ "Kakamega","Kilifi", "Nyeri"]
STATS      = ["mean"]                     # add "p10", "p90" for the uncertainty scenarios (3x the disk space)
IN_DIR     = r"F:\VI_prediction\FORECASTED_WEATHER_BC_V2"    # bias-corrected coarse forecast (output of the SEAS5 bias-correction notebook)
OUT_DIR    = r"F:\VI_prediction\FORECASTED_WEATHER_30M_BC_V2" # NEW folder: Notebook A (FC30_DIR) must point here
# Counties whose 30 m temperature / RH / tmax / tmin are re-centred ON THE FLY, so the county mean of each 30 m file equals the county mean of its
# coarse input file in IN_DIR. CHANGED: now all three counties (was Nyeri only) so every county is treated the same way.
# With the small reference window below the shifts should be tiny (|bias| well under ~0.5 degC); a large printed bias means the reference is still wrong.
RECENTRE_COUNTIES = ["Kakamega", "Kilifi", "Nyeri"]
DEM_PATH   = r"F:\VI_prediction\SRTM\{county}\{county_lower}_dem_clipped.tif"   # standard DEM: F:\VI_prediction\SRTM\Kilifi\kilifi_dem_clipped.tif
# Satellite images of the VI data: one example file per county ({county} is filled in for each county)
S2_PATH      = r"F:\VI_prediction\Sentinel_AllBands_2024\{county}\S2_{county}_2020_01.tif"                      # Sentinel-2 (10 m)
LANDSAT_PATH = r"F:\VI_prediction\LANDSAT_STACK\tiff\{county}\LANDSAT_STACK_{county}_*.tif"            # Landsat stack (30 m); * = first file found is used
# Which image defines the output grid (same CRS, origin and pixel size, so the 30 m weather overlays the satellite data exactly):
#   "landsat"  = 30 m, matches RES_M (recommended)
#   "sentinel" = 10 m, about 9x more pixels and disk space than 30 m
#   None       = automatic UTM grid at RES_M
GRID_FROM   = "landsat"
VI_TEMPLATE = {"landsat": LANDSAT_PATH, "sentinel": S2_PATH}.get(GRID_FROM)
RES_M       = 30                           # used only when GRID_FROM = None
DATE_START, DATE_END = "2026-10-01", "2027-05-03"
# DEM correction parameters
LAPSE_T        = -6.5 / 1000               # degC per metre, air temperature
LAPSE_TD       = -2.0 / 1000               # degC per metre, dewpoint
DEM_SMOOTH_PX  = 3                         # mean filter (pixels at 30 m) to damp canopy/building spikes (ALOS RTC); 1 = off
# CHANGED (was 1.0): reference elevation = DEM averaged to each 0.05 deg forecast cell (no further smoothing), then interpolated to 30 m.
# The 0.05 deg input already carries the coarse terrain signal (cool Aberdare / Mt Kenya / Nandi hills), so only the elevation
# variation INSIDE each coarse cell should be added. A 1 deg window made the reference ~ the county mean and double counted the terrain
# (Nyeri 30 m temperature stretched to -3.5..26 degC from 12.3..18.7 degC). Keep this equal to the coarse cell size (0.05).
REF_WINDOW_DEG = 0.05
REF_DEM_PATH   = None                      # optional UNCLIPPED regional DEM for the reference (better than the county-clipped DEM)
# Rainfall spatial pattern from historical CHIRPS (redistributes the forecast rain, keeps the county-mean amount)
USE_CHIRPS_PATTERN = False
CHIRPS_PATH   = r"F:\historic_weather\CHIRPS_DATA_PY_daily\filled_daily\{county}\CHIRPS_{county}_{year}_{month:02d}_*.tif"
CHIRPS_YEARS  = range(2020, 2026)          # 2020-2025
RATIO_LO, RATIO_HI = 0.5, 2.0              # cap on the local wetter/drier factor
# True so the old files are skipped.
SKIP_EXISTING  = True
# ---- speed knobs (defaults = everything, as before) ----
# RUN_VARS : None = all layers. ["temp", "rh"] = only those two (all that Notebook A / A2 / C need; add "tmax", "tmin" if PestRisk reads them).
#            Skipping rain / wind saves most of the time (they are resampled only).
# DATE_STEP: 1 = every day. 15 = every 15th day (about 15 dates), enough for Notebook A to recover dz and the RH offset.
RUN_VARS = ["temp", "rh", "rain", "wind_speed", "tmax", "tmin"]
DATE_STEP = 15
# (folder group, variable folder/name, kind)
LAYERS = [("temperature", "temp", "temp"), ("humidity", "rh", "rh"),
          ("rainfall", "rain", "plain"), ("wind", "wind_speed", "plain"),
          ("temperature", "tmax", "tx"), ("temperature", "tmin", "tx")]     # tx = daily maximum / minimum temperature: resampled, then the same lapse correction as temp (used by the pest-risk notebook)

## 1. Helpers

In [ ]:
A_MAGNUS, B_MAGNUS = 17.625, 243.04

def rh_from_td(t, td):
    return np.clip(100 * np.exp(A_MAGNUS * td / (B_MAGNUS + td)) / np.exp(A_MAGNUS * t / (B_MAGNUS + t)), 0, 100)

def td_from_rh(t, rh):
    g = np.log(np.clip(rh, 1, 100) / 100) + A_MAGNUS * t / (B_MAGNUS + t)
    return B_MAGNUS * g / (A_MAGNUS - g)

def in_path(county, group, var, stat, date):
    return os.path.join(IN_DIR, group, var, stat, f"{county}_{var}_{stat}_{date}.tif")

def out_path(county, group, var, stat, date):
    return os.path.join(OUT_DIR, group, var, stat, f"{county}_{var}_{stat}_{date}.tif")

def list_dates(county, group, var, stat):
    pat = os.path.join(IN_DIR, group, var, stat, f"{county}_{var}_{stat}_*.tif")
    found = {}
    for p in glob.glob(pat):
        m = re.search(r"_(\d{4}-\d{2}-\d{2})\.tif$", p)          # ignores *_stack.tif
        if m and DATE_START <= m.group(1) <= DATE_END:
            found[m.group(1)] = p
    return sorted(found)

def read_coarse(path):
    with rasterio.open(path) as src:
        a = src.read(1, masked=True).astype("float32").filled(np.nan)
        return a, src.transform, src.crs

def fill_nearest(a):
    """Fill NaN (outside the county polygon) with the nearest valid value so bilinear resampling has no holes at the edges."""
    m = np.isnan(a)
    if not m.any() or m.all():
        return a
    idx = ndimage.distance_transform_edt(m, return_distances=False, return_indices=True)
    return a[tuple(idx)]

def to_grid(a, src_tf, src_crs, G, resampling=Resampling.bilinear):
    dst = np.full(G["shape"], np.nan, dtype="float32")
    reproject(a, dst, src_transform=src_tf, src_crs=src_crs, dst_transform=G["transform"], dst_crs=G["crs"],
              resampling=resampling, src_nodata=np.nan, dst_nodata=np.nan)
    return dst

def nan_smooth(a, size, keep_mask=True):
    """NaN-aware moving average."""
    if size <= 1:
        return a
    m = np.isfinite(a)
    s = ndimage.uniform_filter(np.where(m, a, 0).astype("float64"), size, mode="nearest")
    c = ndimage.uniform_filter(m.astype("float64"), size, mode="nearest")
    with np.errstate(invalid="ignore", divide="ignore"):
        out = s / c
    out[c < 1e-6] = np.nan
    if keep_mask:
        out[~m] = np.nan
    return out.astype("float32")

def open_dem(path):
    d = rxr.open_rasterio(path, masked=True).squeeze(drop=True)
    return d.astype("float32")

def build_grid(county, template_tif):
    """30 m target grid, DEM on that grid, and the smoothed 'coarse' reference elevation on that grid."""
    dem = open_dem(DEM_PATH.format(county=county, county_lower=county.lower()))
    if VI_TEMPLATE:
        tpl_path = VI_TEMPLATE.format(county=county, county_lower=county.lower())
        matches = sorted(glob.glob(tpl_path))                   # tpl_path may contain * (first acquisition date differs per county)
        if not matches: raise FileNotFoundError(f"grid template for {county} not found: {tpl_path}")
        tpl_path = matches[0]
        tpl = rxr.open_rasterio(tpl_path, masked=True)
        if "band" in tpl.dims: tpl = tpl.isel(band=0)          # multi-band stack: only its grid (CRS, origin, pixel size) is used
        tpl = tpl.squeeze(drop=True)
        print(f"{county}: output grid taken from {GRID_FROM} image {os.path.basename(tpl_path)}")
        dem30 = dem.rio.reproject_match(tpl, resampling=Resampling.average)
    else:
        dem30 = dem.rio.reproject(dem.rio.estimate_utm_crs(), resolution=RES_M, resampling=Resampling.average, nodata=np.nan)
    G = {"crs": dem30.rio.crs, "transform": dem30.rio.transform(), "shape": dem30.shape,
         "x": dem30.x.values, "y": dem30.y.values}
    dem30v = nan_smooth(dem30.values.astype("float32"), DEM_SMOOTH_PX)
    valid = np.isfinite(dem30v)

    # reference elevation = DEM averaged to the forecast pixels, then smoothed over ~REF_WINDOW_DEG, then interpolated to 30 m
    with rasterio.open(template_tif) as t:
        ctf, ccrs, cshape, cres = t.transform, t.crs, t.shape, abs(t.res[0])
    rdem = open_dem(REF_DEM_PATH.format(county=county, county_lower=county.lower())) if REF_DEM_PATH else dem
    dem_c = np.full(cshape, np.nan, dtype="float32")
    reproject(rdem.values, dem_c, src_transform=rdem.rio.transform(), src_crs=rdem.rio.crs, dst_transform=ctf, dst_crs=ccrs,
              resampling=Resampling.average, src_nodata=np.nan, dst_nodata=np.nan)
    win = max(1, int(round(REF_WINDOW_DEG / cres)));  win += (win % 2 == 0)
    ref_c = fill_nearest(nan_smooth(dem_c, win, keep_mask=False))
    ref30 = to_grid(ref_c, ctf, ccrs, G)
    G.update(dem=dem30v, valid=valid, ref=ref30, dz=(dem30v - ref30))
    print(f"{county}: grid {G['shape']} @ {G['crs']} | DEM {np.nanmin(dem30v):.0f}-{np.nanmax(dem30v):.0f} m | "
          f"reference {np.nanmean(ref30):.0f} m | dz {np.nanmin(G['dz']):.0f} to {np.nanmax(G['dz']):.0f} m "
          f"-> temp shift {LAPSE_T*np.nanmax(G['dz']):.1f} to {LAPSE_T*np.nanmin(G['dz']):+.1f} degC")
    return G

def write_tif(arr, path, G, **tags):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    arr = np.where(G["valid"], arr, np.nan).astype("float32")
    prof = dict(driver="GTiff", height=arr.shape[0], width=arr.shape[1], count=1, dtype="float32",
                crs=G["crs"], transform=G["transform"], nodata=np.nan, compress="deflate", zlib_level=1, tiled=True, predictor=3)
    tmp = path + ".part"            # write to a temp name first so a failed write never leaves a corrupt .tif
    try:
        with rasterio.open(tmp, "w", **prof) as dst:
            dst.write(arr, 1); dst.update_tags(**tags)
        os.replace(tmp, path)
    except Exception:
        if os.path.exists(tmp): os.remove(tmp)
        raise

def coarse_to_fine(path, G):
    a, tf, crs = read_coarse(path)
    return to_grid(fill_nearest(a), tf, crs, G)

def rain_ratio30(county, month, G):
    """Local wetter/drier factor for a calendar month: CHIRPS monthly climatology / its ~REF_WINDOW_DEG smoothing,
    capped and rescaled to mean 1 so the forecast's county-mean rainfall is preserved. Returned on the 30 m grid."""
    cache = G.setdefault("ratio", {})
    if month in cache:
        return cache[month]
    files = [f for y in CHIRPS_YEARS for f in glob.glob(CHIRPS_PATH.format(county=county, year=y, month=month))]
    if not files:
        print(f"  WARNING no CHIRPS files for {county} month {month}; rain left unpatterned"); cache[month] = None; return None
    acc, n = None, None
    for f in files:
        with rasterio.open(f) as s:
            a = s.read(1, masked=True).astype("float32").filled(np.nan); tf, crs = s.transform, s.crs; res = abs(s.res[0])
        a[a < 0] = np.nan
        ok = np.isfinite(a)
        acc = np.where(ok, a, 0) if acc is None else acc + np.where(ok, a, 0)
        n = ok.astype("float32") if n is None else n + ok
    clim = np.where(n > 0, acc / np.maximum(n, 1), np.nan).astype("float32")          # mean mm/day for that month
    win = max(1, int(round(REF_WINDOW_DEG / res))); win += (win % 2 == 0)
    m = np.isfinite(clim)
    num = ndimage.uniform_filter(np.where(m, clim, 0).astype("float64"), win, mode="nearest")
    den = ndimage.uniform_filter(m.astype("float64"), win, mode="nearest")
    smooth = num / np.maximum(den, 1e-6)
    ratio = np.clip(clim / np.maximum(smooth, 0.1), RATIO_LO, RATIO_HI).astype("float32")
    ratio[~m] = np.nan
    ratio = ratio / np.nanmean(ratio)                                                    # keep the county-mean forecast amount
    r30 = to_grid(fill_nearest(ratio.astype("float32")), tf, crs, G)
    cache[month] = r30
    print(f"  CHIRPS pattern month {month}: {len(files)} files, ratio {np.nanmin(r30):.2f}-{np.nanmax(r30):.2f}")
    return r30

## 2. Process one county / stat
Temperature and RH are done together because RH needs the corrected temperature. If any input day is missing for a variable, a warning is printed and that variable is skipped for that day.

In [ ]:
import glob, os
for p in glob.glob(os.path.join(OUT_DIR, "**", "*.part"), recursive=True):
    os.remove(p); print("removed", p)

In [ ]:
def process(county, stat):
    dates = list_dates(county, "temperature", "temp", stat)[::DATE_STEP]
    if not dates:
        print(f"WARNING no temp files for {county}/{stat} in {DATE_START}..{DATE_END}"); return
    n_done = sum(os.path.exists(out_path(county, "temperature", "temp", stat, d)) and
                 os.path.exists(out_path(county, "humidity", "rh", stat, d)) for d in dates)
    print(f"{county} {stat}: {len(dates)} days to process, {n_done} already done (will be skipped)", flush=True)
    if county not in GRIDS or GRIDS[county] is None:                      # the 30 m grid / dz is the same for every stat: build it once per county
        print(f"{county}: building 30 m grid (can take a minute or two)...", flush=True)
        GRIDS[county] = build_grid(county, in_path(county, "temperature", "temp", stat, dates[0]))
    G = GRIDS[county]
    want_tr = (not RUN_VARS) or ("temp" in RUN_VARS and "rh" in RUN_VARS)
    dz = G["dz"]
    for i, d in enumerate(dates, 1):
        done = lambda g, v: SKIP_EXISTING and os.path.exists(out_path(county, g, v, stat, d))
        have = lambda g, v: os.path.exists(in_path(county, g, v, stat, d))

        if want_tr and have("temperature", "temp") and have("humidity", "rh") and not (done("temperature", "temp") and done("humidity", "rh")):
            T  = coarse_to_fine(in_path(county, "temperature", "temp", stat, d), G)
            RH = coarse_to_fine(in_path(county, "humidity", "rh", stat, d), G)
            Td = td_from_rh(T, RH); del RH                                     # free big arrays as soon as they are used (memory)
            Tc = T + LAPSE_T * dz;  del T
            Tdc = Td + LAPSE_TD * dz; del Td
            tg, how_t, how_r = {}, "DEM lapse", "DEM T/Td lapse"
            if county in RECENTRE_COUNTIES:                                    # remove the county-mean shift of the lapse step (mean of the 30 m field vs mean of the coarse input)
                c0 = float(np.nanmean(read_coarse(in_path(county, "temperature", "temp", stat, d))[0]))
                bias = float(np.nanmean(np.where(G["valid"], Tc, np.nan)) - c0)
                Tdc = td_from_rh(Tc, rh_from_td(Tc, Tdc)) - bias * (LAPSE_TD / LAPSE_T)   # dewpoint moves by the matching amount (uses the unshifted Tc)
                Tc = Tc - bias
                tg, how_t, how_r = {"recentered": "yes", "recentre_bias_degC": f"{bias:.3f}"}, "DEM lapse + re-centred", "DEM T/Td lapse + re-centred"
            write_tif(Tc, out_path(county, "temperature", "temp", stat, d), G, units="degC", variable="temp", stat=stat, correction=how_t, **tg)
            write_tif(rh_from_td(Tc, Tdc), out_path(county, "humidity", "rh", stat, d), G, units="percent", variable="rh", stat=stat, correction=how_r, **tg)
            del Tc, Tdc
        elif want_tr and not (have("temperature", "temp") and have("humidity", "rh")):
            print(f"  missing temp/rh input for {county} {stat} {d}")

        for g, v, kind in LAYERS:
            if kind in ("temp", "rh"): continue
            if RUN_VARS and v not in RUN_VARS: continue
            if not have(g, v): print(f"  missing {v} input for {county} {stat} {d}"); continue
            if done(g, v): continue
            arr = coarse_to_fine(in_path(county, g, v, stat, d), G)
            how = "resample only"
            xt = {}
            if kind == "tx":
                if np.isfinite(np.nanmedian(arr)) and np.nanmedian(arr) > 150: arr = arr - 273.15      # Kelvin -> degC
                arr, how = arr + LAPSE_T * dz, "DEM lapse"
                if county in RECENTRE_COUNTIES:
                    ca = read_coarse(in_path(county, g, v, stat, d))[0]
                    if np.isfinite(np.nanmedian(ca)) and np.nanmedian(ca) > 150: ca = ca - 273.15
                    shift = float(np.nanmean(np.where(G["valid"], arr, np.nan)) - np.nanmean(ca))
                    arr, how = arr - shift, "DEM lapse + re-centred"
                    xt = {"recentered": "yes", "recentre_bias_degC": f"{shift:.3f}"}
            if v == "rain" and USE_CHIRPS_PATTERN:
                r = rain_ratio30(county, int(d[5:7]), G)
                if r is not None:
                    arr, how = arr * r, "CHIRPS monthly pattern"
            write_tif(arr, out_path(county, g, v, stat, d), G, variable=v, stat=stat, correction=how, **({"units": "degC"} if kind == "tx" else {}), **xt)
        gc.collect()                                                           # release memory after every day
        if i % 10 == 0 or i == len(dates):
            print(f"  {county} {stat}: {i}/{len(dates)} days ({d})", flush=True)
    return G

GRIDS = {}
for county in COUNTIES:
    for stat in STATS:
        process(county, stat)
    if GRIDS.get(county):                                                      # county finished: keep only what the quick check needs (dz), free the rest
        for k in ("dem", "ref", "ratio", "x", "y"): GRIDS[county].pop(k, None)
    gc.collect()
print("ALL DONE", flush=True)

In [ ]:
import os, glob
base = os.path.join(IN_DIR, "temperature", "temp", "mean")
print("folder exists:", os.path.isdir(base))
for c in COUNTIES:
    fs = sorted(glob.glob(os.path.join(base, f"{c}_temp_mean_*.tif")))
    print(c, len(fs), fs[0][-14:] if fs else None, fs[-1][-14:] if fs else None)

In [ ]:
# ===== RAIN ONLY: redo rainfall with the CHIRPS spatial pattern =====
# Needs the cells above to have run (settings, helpers, process). Only rainfall is rewritten.
import rasterio, numpy as np

_old = dict(RUN_VARS=RUN_VARS, SKIP_EXISTING=SKIP_EXISTING)
USE_CHIRPS_PATTERN = True            # switch the pattern on
RUN_VARS = ["rain"]                  # only rainfall
SKIP_EXISTING = False                # overwrite the plain-resampled rain files

try:
    for county in COUNTIES:
        for stat in STATS:
            process(county, stat)
        gc.collect()
finally:
    RUN_VARS = _old["RUN_VARS"]; SKIP_EXISTING = _old["SKIP_EXISTING"]   # restore

# ---- check: pattern applied, and county-mean rain unchanged vs the coarse forecast ----
rows = []
for county in COUNTIES:
    for stat in STATS:
        ds = list_dates(county, "temperature", "temp", stat)[::DATE_STEP]
        for d in ds[::max(1, len(ds) // 4)]:
            pi, po = in_path(county, "rainfall", "rain", stat, d), out_path(county, "rainfall", "rain", stat, d)
            if not (os.path.exists(pi) and os.path.exists(po)): continue
            a = read_coarse(pi)[0]
            with rasterio.open(po) as s:
                b = s.read(1, masked=True).astype("float32").filled(np.nan); how = s.tags().get("correction")
            rows.append(dict(county=county, date=d, coarse_mean=np.nanmean(a), fine_mean=np.nanmean(b),
                             fine_min=np.nanmin(b), fine_max=np.nanmax(b), correction=how))
print(pd.DataFrame(rows).round(2).to_string())

## 3. Quick check
Left: original 0.05° forecast. Middle: corrected 30 m. Right: the elevation offset (DEM minus reference) that drove the correction.

In [ ]:
def qc(county, stat="mean", date=None):
    dates = list_dates(county, "temperature", "temp", stat)[::DATE_STEP]; date = date or dates[len(dates) // 2]   # only dates the main run wrote
    G = GRIDS[county]
    a, tf, crs = read_coarse(in_path(county, "temperature", "temp", stat, date))
    with rasterio.open(out_path(county, "temperature", "temp", stat, date)) as s: b = s.read(1, masked=True)
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
    im0 = ax[0].imshow(a, cmap="coolwarm"); ax[0].set_title(f"{county} temp {date} - original (0.05 deg)"); plt.colorbar(im0, ax=ax[0], shrink=.8)
    im1 = ax[1].imshow(b, cmap="coolwarm"); ax[1].set_title("corrected 30 m (degC)"); plt.colorbar(im1, ax=ax[1], shrink=.8)
    im2 = ax[2].imshow(G["dz"], cmap="terrain"); ax[2].set_title("DEM - reference (m)"); plt.colorbar(im2, ax=ax[2], shrink=.8)
    for x in ax: x.axis("off")
    plt.tight_layout(); plt.show()

for county in GRIDS: qc(county)

## Notes
- **Reference elevation.** The forecast represents a smooth, coarse surface, so the correction uses `DEM(30 m) − reference`, where the reference is the DEM averaged and smoothed over ~`REF_WINDOW_DEG`. A county-clipped DEM can't give a true 1° cell mean; if you have an unclipped regional DEM, set `REF_DEM_PATH` for a better reference. If the corrected county mean looks too warm or cool compared with the original, this is the setting to look at.
- **Lapse rates.** −6.5 °C/km (temperature) and −2 °C/km (dewpoint) are standard values. Estimating them locally from ERA5-Land vs elevation is better, especially for coastal Kilifi.
- **RH** is derived from the *daily mean* RH and temperature, so it is an approximation of the mean of the 6-hourly RH. The direction of the elevation effect (higher = more humid) is right.
- **Disk space:** 30 m is large (Kilifi is ~13 million pixels per day per variable). Keep `STATS = ["mean"]` unless you need the p10/p90 scenarios.
- **Rainfall pattern (CHIRPS).** The factor is the CHIRPS monthly mean divided by its ~1° smoothing, capped at 0.5-2 and rescaled to mean 1. It only redistributes the forecast rain inside the county; it does not bias-correct the amount (that needs the SEAS5 rain archive). Six years is a short climatology, so the pattern is noisy. It is CHIRPS-resolution (~5 km), not 30 m detail. It is currently switched off (`USE_CHIRPS_PATTERN = False`): rain is resampled only.
- **ALOS PALSAR RTC** can include canopy height; `DEM_SMOOTH_PX` reduces the false spikes.
- **Grid from Landsat / Sentinel-2.** `GRID_FROM = "landsat"` puts the 30 m output on the grid of the first Landsat stack of each county (`LANDSAT_PATH`), so the corrected weather lines up pixel by pixel with that imagery; only the grid of the file is used, not its bands. `"sentinel"` uses the Sentinel-2 file (10 m: about 9 times more pixels, so Kilifi would be roughly 100 million pixels per day and variable). `None` goes back to the automatic UTM grid. The grid of the weather, `dz` and the stores of Notebook A / A2 must be the grid of the VI files that A uses as its reference: if those VI files are not on the Landsat grid, set `GRID_FROM` to match them. Delete the old files in `OUT_DIR` before re-running, because `SKIP_EXISTING` would keep files written on the old grid.


In [ ]:
# ===== CHECK: county mean of each 30 m file vs its coarse input (sample of dates) =====
# Re-centred counties (RECENTRE_COUNTIES) should show ~0 for temp, tmax, tmin. Other counties keep the small shift from the lapse step.
# The 'recentered' tag shows which files were re-centred on the fly. Uses IN_DIR / OUT_DIR / STATS from the settings cell: nothing to edit here.
rows = []
for county in COUNTIES:
    for stat in STATS:
        ds = list_dates(county, "temperature", "temp", stat)[::DATE_STEP]          # only dates the main run wrote
        for d in ds[::max(1, len(ds) // 6)]:
            for g, v in [("temperature", "temp"), ("temperature", "tmax"), ("temperature", "tmin"), ("humidity", "rh")]:
                pi, po = in_path(county, g, v, stat, d), out_path(county, g, v, stat, d)
                if not (os.path.exists(pi) and os.path.exists(po)): continue
                a = read_coarse(pi)[0]
                if v != "rh" and np.nanmedian(a) > 150: a = a - 273.15
                with rasterio.open(po) as s:
                    b = s.read(1, masked=True).astype("float32").filled(np.nan); tag = s.tags().get("recentered") == "yes"
                rows.append(dict(county=county, stat=stat, var=v, date=d, shift=float(np.nanmean(b) - np.nanmean(a)), recentred=tag))
chk = pd.DataFrame(rows)
if len(chk):
    print(chk.groupby(["county", "stat", "var"]).agg(shift_mean=("shift", "mean"), shift_min=("shift", "min"), shift_max=("shift", "max"), recentred=("recentred", "all")).round(2).to_string())
    bad = chk[chk.county.isin(RECENTRE_COUNTIES) & (~chk.recentred)]
    print("\nRE-CENTRED COUNTIES with untagged files (delete OUT_DIR for them and re-run):", len(bad))


In [ ]:
# ===== CHECK: RH saturation in the 30 m files (share of pixels at/near 100 %) vs the coarse input =====
# Uses IN_DIR / OUT_DIR / STATS / COUNTIES / DATE_STEP from the settings cell. Nothing to edit.
rows = []
for county in COUNTIES:
    for stat in STATS:
        ds = list_dates(county, "temperature", "temp", stat)[::DATE_STEP]
        for d in ds[::max(1, len(ds) // 6)]:                       # same sample of dates as the mean-shift check
            pi, po = in_path(county, "humidity", "rh", stat, d), out_path(county, "humidity", "rh", stat, d)
            if not (os.path.exists(pi) and os.path.exists(po)): continue
            a = read_coarse(pi)[0]
            with rasterio.open(po) as s:
                b = s.read(1, masked=True).astype("float32").filled(np.nan)
            fa, fb = np.isfinite(a), np.isfinite(b)
            rows.append(dict(county=county, date=d,
                             coarse_mean=float(np.nanmean(a)), coarse_max=float(np.nanmax(a)),
                             coarse_pct_ge99=100 * float(np.mean(a[fa] >= 99)),
                             fine_mean=float(np.nanmean(b)), fine_min=float(np.nanmin(b)), fine_max=float(np.nanmax(b)),
                             fine_pct_ge99=100 * float(np.mean(b[fb] >= 99)),
                             fine_pct_eq100=100 * float(np.mean(b[fb] >= 99.999))))
rh = pd.DataFrame(rows)
print(rh.groupby("county").agg(coarse_mean=("coarse_mean", "mean"), coarse_pct_ge99=("coarse_pct_ge99", "mean"),
                               fine_mean=("fine_mean", "mean"), fine_min=("fine_min", "min"), fine_max=("fine_max", "max"),
                               fine_pct_ge99=("fine_pct_ge99", "mean"), fine_pct_eq100=("fine_pct_eq100", "mean"),
                               worst_pct_ge99=("fine_pct_ge99", "max")).round(2).to_string())
# Reading it: fine_pct_ge99 should be close to coarse_pct_ge99. A few percent more is fine;
# tens of percent more (worst_pct_ge99 high) means the dewpoint lapse is saturating the mountains.